# **R/S BENCHMARK - DATASET GENERATION**

## **1. State limit function**

$$g = k(t) \cdot \frac{R}{z_1} - S \cdot z_2$$

$z_1$ and $z_2$ are normal latent multipliers (mean 1.0, sd 0.028 and 0.096); $k(t) = 1 +
(k_{final}-1)\,t/100$ is the degradation factor.

## **2. Libraries**

In [9]:
import sys
import time
from pathlib import Path

# functions.py sits one directory up
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Normal, JointIndependent

## **3. Random variables and fixed parameters**

Design variables $R$ and $S$, plus everything the emulator needs that isn't a design variable.

In [10]:
r_mean = 5.0   # resistance mean
r_std  = 0.8   # resistance standard deviation
s_mean = 2.0   # load mean
s_std  = 0.6   # load standard deviation

n_samples            = 500      # Number of design samples
n_latent_samples     = 2500     # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 250      # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)
k_factor_final       = 0.3      # Degradation factor at t = 100. Use 1.0 for no time effect
z1_std               = 0.028    # Standard deviation of the resistance latent multiplier
z2_std               = 0.096    # Standard deviation of the load latent multiplier

times = np.linspace(0, 150, 10, endpoint=True)  # Time points for the degradation factor
times

array([  0.        ,  16.66666667,  33.33333333,  50.        ,
        66.66666667,  83.33333333, 100.        , 116.66666667,
       133.33333333, 150.        ])

## **4. Design samples**

In [11]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

x_pce_rvs = joint.rvs(n_samples)
x_val     = joint.rvs(n_samples_validation)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")

Samples generated successfully!
   Number of design samples: 500
   Number of latent samples per design sample: 2500
   Total simulations per time step: 1875000


In [12]:
x_pce_rvs

array([[5.92906252, 2.22090468],
       [6.10058584, 1.70551362],
       [5.23732651, 2.50814765],
       [3.47797492, 1.31164008],
       [4.89049805, 3.02367055],
       [6.4604523 , 2.22012087],
       [5.06912583, 2.36021269],
       [5.43983135, 2.61627484],
       [5.19481648, 2.10723675],
       [5.71725558, 2.72528253],
       [4.40337173, 1.82453998],
       [4.8841755 , 2.87008946],
       [4.12214107, 0.97598607],
       [4.71965712, 1.92499801],
       [5.98414402, 1.68480644],
       [4.90766539, 0.78010431],
       [6.29624349, 1.87324247],
       [4.3866536 , 2.18352531],
       [4.60153038, 1.68303759],
       [4.28363016, 2.1185159 ],
       [4.0701228 , 2.32963807],
       [5.19239402, 1.11523508],
       [2.99231019, 2.11190738],
       [4.619237  , 1.41423019],
       [5.73667703, 2.46914496],
       [4.6216366 , 1.9633026 ],
       [4.72669977, 1.65084482],
       [3.99489659, 2.63694391],
       [5.30104705, 2.55281954],
       [5.26171119, 2.3571274 ],
       [5.

## **5. Generate the dataset at each time step**

Steps:

- $g$ evaluation;
- GLD fit; and
- saving `dataset_full`/`dataset_unique` for both splits.

In [13]:
print("="*60)
print("GENERATING THE BENCHMARK DATASET")
print("="*60)

generation_results = []
for t in times:
    result = generate_dataset_at_time_benchmark(
                                                   x_train=x_pce_rvs,
                                                   x_val=x_val,
                                                   time_step=t,
                                                   n_latent_samples=n_latent_samples,
                                                   k_factor_final=k_factor_final,
                                                   z1_std=z1_std,
                                                   z2_std=z2_std,
                                                   output_dir='.',
                                               )
    generation_results.append(result)

GENERATING THE BENCHMARK DATASET

----------------------------------------
GENERATING DATASET FOR TIME STEP: 0.0 years
----------------------------------------
  train: 500 design points, 2.53 s total
  val: 250 design points, 1.26 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 16.666666666666668 years
----------------------------------------
  train: 500 design points, 2.44 s total
  val: 250 design points, 1.24 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 33.333333333333336 years
----------------------------------------
  train: 500 design points, 2.53 s total
  val: 250 design points, 1.26 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 50.0 years
----------------------------------------
  train: 500 design points, 2.40 s total
  val: 250 design points, 1.23 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 66.66666666666667 years
--------

In [18]:
with open('2500_dataset_full_train_0.0_benchmark.pkl', 'rb') as f:
    obj = dill.load(f)
obj.head()

,r,s,z1_latent,R_effective,z2_latent,S_effective,k factor,Time (years),g,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
0,5.929063,2.220905,1.049087,5.651640,0.920444,2.044217,1.0,0.0,3.607423,3.700467,5.619118,0.142716,0.115782,0.010921
1,5.929063,2.220905,1.015428,5.838981,0.925003,2.054344,1.0,0.0,3.784637,3.700467,5.619118,0.142716,0.115782,0.010921
2,5.929063,2.220905,1.030655,5.752715,0.974502,2.164276,1.0,0.0,3.588439,3.700467,5.619118,0.142716,0.115782,0.010921
3,5.929063,2.220905,1.018461,5.821589,0.972842,2.160590,1.0,0.0,3.660999,3.700467,5.619118,0.142716,0.115782,0.010921
4,5.929063,2.220905,1.039047,5.706250,0.923302,2.050566,1.0,0.0,3.655684,3.700467,5.619118,0.142716,0.115782,0.010921


## 6. Timing summary

Cost of building the dataset, per time step.

In [ ]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_benchmark.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

train_total = emulator_timing['Train total (s)'].sum()
val_total   = emulator_timing['Val total (s)'].sum()

print(f"Train split - eg-value dataset generation time: {train_total:.1f} s")
print(f"Val split   - g-value dataset generation time: {val_total:.1f} s")
print(f"Total g-value dataset generation time (train + val): {train_total + val_total:.1f} s")
emulator_timing